**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher

# Example Chapter 14: Simulating Returns
In this notebook we simulate monthly returns for many stocks and combine them into a portfolio. Every stock is exposed to the same source of **systematic risk** (the tangent portfolio) and also has its own **idiosyncratic risk**. The simulation shows that when a portfolio holds many stocks, idiosyncratic risk is diversified away, but systematic risk is not.

### Imports and Setup

In [ ]:
# Import packages
%pip install -q --upgrade farms

import farms as fm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Establish display options when printing DataFrames
pd.set_option('display.max_columns', None)   # Show all columns without truncation
pd.set_option('display.width', 1000)   # Set the display width so output stays on one line
pd.set_option('display.max_rows', 20)   # Force truncation if DataFrame has more than 20 rows

# Create a random number generator. Fixing the seed means you get the same
# simulated returns every time you run the notebook.
rng = np.random.default_rng(seed=13)

### Choose the Simulation Settings
All of the assumptions for the simulation are set in this one cell, so it is easy to change them later.

- `N` is the number of stocks. More stocks means more diversification.
- `M` is the number of months. More months means less sampling error in our estimates.
- `mu_T` and `sigma_T` are the monthly expected excess return and volatility of the tangent portfolio.
- `beta` is the beta of every stock relative to the tangent portfolio.
- `z_vol_low` and `z_vol_high` set the range for each stock's idiosyncratic volatility.

In [ ]:
N = 2   # number of stocks
M = 1000   # number of months

mu_T = 0.004      # tangent portfolio expected excess return: 0.4% per month (4.8% per year)
sigma_T = 0.06    # tangent portfolio volatility: 6% per month (about 20.8% per year)

beta = 1.0        # every stock has the same beta

z_vol_low = 0.05  # smallest idiosyncratic volatility (5% per month)
z_vol_high = 0.15 # largest idiosyncratic volatility (15% per month)

### Simulate Tangent Portfolio Returns
We draw `M` monthly excess returns for the tangent portfolio from a normal distribution with mean `mu_T` and standard deviation `sigma_T`. The function `rng.normal(mean, std, size)` returns an array of random draws.

In [ ]:
tangent_returns = rng.normal(mu_T, sigma_T, M)

print('First five tangent portfolio returns:', tangent_returns[:5])

### Simulate Stock Returns
Each stock's excess return is

$$r_i - r_f = \beta_i \, r_T + z_i$$

where $r_T$ is the tangent portfolio excess return and $z_i$ is an **idiosyncratic shock** that affects only stock $i$. The variance of $z_i$ is the stock's idiosyncratic risk.

The code works in three steps:

1. `z_vol` gives each stock its own idiosyncratic volatility, drawn uniformly between `z_vol_low` and `z_vol_high`.
2. `shocks` is a table with `M` rows (months) and `N` columns (stocks). Because `z_vol` has one value per stock, NumPy uses `z_vol[i]` as the standard deviation for column `i`.
3. `tangent_returns.reshape(M, 1)` turns the tangent returns into a single column, so NumPy adds `beta` times that column to every stock's shocks.

Finally, we store the results in a DataFrame with one column per stock, numbered 1 to `N`.

In [ ]:
# Step 1: idiosyncratic volatility for each stock
z_vol = rng.uniform(z_vol_low, z_vol_high, N)

# Step 2: idiosyncratic shocks, one row per month and one column per stock
shocks = rng.normal(0, z_vol, size=(M, N))

# Step 3: stock excess returns = beta * tangent return + shock
stock_returns = beta * tangent_returns.reshape(M, 1) + shocks

stock_returns = pd.DataFrame(stock_returns, columns=range(1, N + 1))

print('Rows (months), columns (stocks):', stock_returns.shape)

# Show the first five months for the first five stocks
stock_returns.iloc[:5, :5]

### Form an Equally Weighted Portfolio
An equally weighted portfolio puts weight $1/N$ on every stock, so its return each month is simply the average return across all stocks. The command `stock_returns.mean(axis=1)` takes the average across the columns (stocks) for each row (month).

We then collect three return series in one DataFrame: one individual stock, the portfolio, and the tangent portfolio.

In [ ]:
portfolio_returns = stock_returns.mean(axis=1)

returns = pd.DataFrame({
    'Stock 1': stock_returns[1],
    'Portfolio': portfolio_returns,
    'Tangent': tangent_returns,
})
returns.head()

### Plot the Returns
The first plot shows the returns of every stock (the colored lines) and the equally weighted portfolio (the thin black line) for every simulated month. `plt.plot(stock_returns)` draws one line for each column of the DataFrame, so we get one colored line per stock. The individual stocks swing widely, but the portfolio stays in a narrow band because the idiosyncratic shocks cancel out across many stocks.

The second plot compares the full distribution of each series using `fm.plot_return_histograms()`. For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#plot_return_histograms).

`plot_return_histograms()`

**Required Inputs**

- `returns` — DataFrame containing numeric decimal-period returns. Example: `returns=returns`.

**Optional Inputs**

- `columns` — Default is `None`, which plots all columns. Example: `columns=['Stock 1', 'Portfolio']`.
- `bins` — Default is `'auto'`. Sets the number of histogram bins. Example: `bins=40`.
- `show_mean` — Default is `True`. Draws a vertical line at each column's mean. Example: `show_mean=False`.
- `sharey` — Default is `True`. Uses one y-axis scale for all panels. Example: `sharey=False`.
- `figsize` — Default is `None`. Sets the overall figure size. Example: `figsize=(12, 4)`.
- `title` — Default is `None`. Sets a figure-level title. Example: `title='Return distributions'`.
- `ncols` — Default is `3`. Sets the number of panels per row. Example: `ncols=2`.

**Outputs**

- `tuple` — Returns `(figure, axes)` containing the Matplotlib Figure and Axes objects.

In [ ]:
# Plot every stock (colored lines) and the portfolio (thin black line)
plt.figure(figsize=(14, 7))
plt.plot(stock_returns, alpha=0.5)
plt.plot(portfolio_returns, color='black', linewidth=1)
plt.title('Simulated Stock Returns and the Equally Weighted Portfolio')
plt.xlabel('Months', fontsize=16)
plt.ylabel('Excess Return', fontsize=16)
plt.show()

# Distribution of returns over all months
fig, axes = fm.plot_return_histograms(
    returns=returns,
    bins=40,
    title='Distribution of Simulated Monthly Returns',
)
plt.show()

### Summary Statistics
We use `fm.summary_stats()` to report the annualized mean and volatility of each series. The portfolio and the tangent portfolio have nearly the same mean and volatility. The single stock is much more volatile, and because of that extra noise its sample mean can be far from the true expected return.

For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#summary_stats).

`summary_stats()`

**Required Inputs**

- `returns` — DataFrame containing numeric decimal-period returns. Example: `returns=returns`.

**Optional Inputs**

- `risk_free` — Default is `None`. When supplied, adds the annualized excess mean and Sharpe ratio. Example: `risk_free=0.002`.
- `frequency` — Default is `'monthly'`. Selects the annualization frequency: `'daily'`, `'weekly'`, or `'monthly'`. Example: `frequency='monthly'`.

**Outputs**

- `DataFrame` — One row per return column with `Observations`, `Annualized arithmetic mean`, and `Annualized volatility`.

In [ ]:
fm.summary_stats(returns=returns, frequency='monthly')

### Split Variance into Systematic and Idiosyncratic Risk
The variance of any stock or portfolio $p$ can be split into two parts:

$$Var[r_p] = \beta_{pT}^2 \, Var[r_T] + Var[z_p]$$

The first term is **systematic risk** and the second is **idiosyncratic risk**. We estimate $\beta_{pT}$ by regressing returns on the tangent portfolio with `fm.slope()`, compute systematic risk, and then find idiosyncratic risk as whatever variance is left over.

For the single stock, idiosyncratic risk is a large share of total variance. For the portfolio it is almost zero: the shocks are independent across stocks, so they cancel out when we average over many stocks. The portfolio beta, however, stays at `beta`, so systematic risk does not diversify away.

For full parameter details, see the [`farms` API documentation](https://github.com/boyerb/farms/blob/main/docs/api.md#slope).

`slope()`

**Required Inputs**

- `y` — Dependent-variable observations. Example: `y=returns['Portfolio']`.
- `x` — Independent-variable observations. Example: `x=returns['Tangent']`.

**Optional Inputs**

- None.

**Outputs**

- `float` — Estimated regression slope (the beta).

In [ ]:
var_T = returns['Tangent'].var()

# Single stock
beta_stock = fm.slope(y=returns['Stock 1'], x=returns['Tangent'])
total_stock = returns['Stock 1'].var()
systematic_stock = beta_stock**2 * var_T   # **2 means "squared"
idiosyncratic_stock = total_stock - systematic_stock

# Equally weighted portfolio
beta_port = fm.slope(y=returns['Portfolio'], x=returns['Tangent'])
total_port = returns['Portfolio'].var()
systematic_port = beta_port**2 * var_T
idiosyncratic_port = total_port - systematic_port

risk = pd.DataFrame(
    {
        'Beta': [beta_stock, beta_port],
        'Total variance': [total_stock, total_port],
        'Systematic risk': [systematic_stock, systematic_port],
        'Idiosyncratic risk': [idiosyncratic_stock, idiosyncratic_port],
    },
    index=['Stock 1', 'Portfolio'],
)
risk['Idiosyncratic share'] = risk['Idiosyncratic risk'] / risk['Total variance']
risk.round(6)

### Try it
1. Set `N = 1`, then `N = 10`, then `N = 100`, and rerun the notebook. How does the portfolio's idiosyncratic share change as you add stocks?
2. Change `beta` to `0.5` and then `1.5`. Does diversification reduce systematic risk?
3. Give each stock a different beta by replacing `beta = 1.0` with `beta = rng.uniform(0.5, 1.5, N)`. What is the portfolio beta, and how does it relate to the average stock beta?
4. Raise `z_vol_high` to `0.40`. How many stocks do you need before the portfolio's idiosyncratic risk is close to zero?
5. Set `M = 60` (five years of data). How much do the estimated betas and variances move around when you change the `seed`?